In [1]:
# =============================================================================
# 记忆组织模块 - Memory Organization Module
# 本代码用于书籍《Memory Engineering》第11章配套代码
# 涵盖：时间结构、事件结构、语义结构、层级抽象、关系建模
# =============================================================================

# %% [markdown]
# # 记忆组织：让零散经验形成可推理的结构
# 
# 本notebook将分模块演示如何将零散的记忆节点组织成可推理的结构化网络。
# 每个模块可独立运行，也可承接前序模块的输出。

# %% [markdown]
# ## 0. 环境设置与依赖声明

# %%
# Setup: environment & reproducibility / 环境与可复现
import sys
import platform
import random
import numpy as np
from datetime import datetime, timedelta
import json
import uuid
from typing import List, Dict, Any, Optional, Tuple, Set
from dataclasses import dataclass, field, asdict
from enum import Enum
import hashlib
from collections import defaultdict

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("=" * 60)

# %% [markdown]
# ## 1. 基础数据结构定义

# %%
# =============================================================================
# 1. 基础数据结构定义 / Base Data Structures
# =============================================================================

class MemoryType(Enum):
    """记忆类型枚举"""
    USER_MEMORY = "UserMemory"
    SYSTEM_MEMORY = "SystemMemory"
    FACT = "fact"
    EVENT = "event"
    PREFERENCE = "preference"


class MemoryStatus(Enum):
    """记忆状态枚举"""
    ACTIVATED = "activated"
    ARCHIVED = "archived"
    DEPRECATED = "deprecated"


class RelationType(Enum):
    """关系类型枚举"""
    CAUSES = "causes"           # 因果关系
    FOLLOWS = "follows"         # 时间顺序
    RESOLVES = "resolves"       # 解决关系
    CONTAINS = "contains"       # 包含关系
    RELATED_TO = "related_to"   # 语义相关
    SAME_TOPIC = "same_topic"   # 同主题
    DEPENDS_ON = "depends_on"   # 依赖关系


@dataclass
class MemoryNode:
    """
    记忆节点：系统的基本存储单元
    
    Attributes:
        id: 唯一标识符
        key: 记忆关键词/标题
        memory: 记忆内容摘要
        tags: 标签列表
        type: 记忆类型
        confidence: 置信度 (0-1)
        created_at: 创建时间
        updated_at: 更新时间
        user_id: 用户ID
        session_id: 会话ID
        background: 背景信息
        source_type: 来源类型
        embedding: 向量表示（可选）
    """
    id: str
    key: str
    memory: str
    tags: List[str]
    type: str
    confidence: float
    created_at: str
    updated_at: str
    user_id: str
    session_id: str
    background: str = ""
    source_type: str = "unknown"
    embedding: Optional[List[float]] = None
    
    def to_dict(self) -> Dict:
        """转换为字典"""
        return asdict(self)
    
    @classmethod
    def from_dict(cls, data: Dict) -> 'MemoryNode':
        """从字典创建"""
        return cls(**{k: v for k, v in data.items() if k in cls.__dataclass_fields__})
    
    def get_timestamp(self) -> datetime:
        """获取创建时间的datetime对象"""
        return datetime.fromisoformat(self.created_at.replace('Z', '+00:00'))


@dataclass
class EventUnit:
    """
    事件单元：结构化的事件表示
    
    Attributes:
        event_id: 事件唯一标识
        agent: 执行主体
        action: 动作
        object: 作用对象
        outcome: 结果
        context: 上下文
        timestamp: 时间戳
        source_memory_id: 来源记忆ID
        confidence: 置信度
    """
    event_id: str
    agent: Optional[str]
    action: str
    object: Optional[str]
    outcome: Optional[str]
    context: Optional[str]
    timestamp: str
    source_memory_id: str
    confidence: float = 0.9


@dataclass
class Edge:
    """
    边：连接两个节点的关系
    
    Attributes:
        source_id: 源节点ID
        target_id: 目标节点ID
        relation_type: 关系类型
        weight: 权重/置信度
        metadata: 额外元数据
    """
    source_id: str
    target_id: str
    relation_type: RelationType
    weight: float = 1.0
    metadata: Dict = field(default_factory=dict)


@dataclass
class Session:
    """
    会话/片段：时间连续的记忆集合
    
    Attributes:
        session_id: 会话ID
        memory_ids: 包含的记忆ID列表
        start_time: 开始时间
        end_time: 结束时间
        topic: 会话主题
    """
    session_id: str
    memory_ids: List[str]
    start_time: str
    end_time: str
    topic: Optional[str] = None


@dataclass
class Phase:
    """
    阶段：更高层次的时间划分
    
    Attributes:
        phase_id: 阶段ID
        label: 阶段标签
        session_ids: 包含的会话ID列表
        start_time: 开始时间
        end_time: 结束时间
        summary: 阶段摘要
    """
    phase_id: str
    label: str
    session_ids: List[str]
    start_time: str
    end_time: str
    summary: Optional[str] = None


@dataclass
class AbstractionNode:
    """
    抽象节点：从具体案例中提炼的模式
    
    Attributes:
        node_id: 节点ID
        label: 模式名称
        condition: 适用条件
        solution: 建议方案
        verification: 验证方法
        support_ids: 支撑案例ID列表
        confidence: 置信度
    """
    node_id: str
    label: str
    condition: str
    solution: str
    verification: str
    support_ids: List[str]
    confidence: float = 0.8


# %%
# 配置类 / Configuration
@dataclass
class MemoryOrganizationConfig:
    """记忆组织配置"""
    # 时间结构参数
    time_threshold_minutes: int = 30      # 会话切分的时间阈值
    topic_drift_threshold: float = 0.3    # 主题漂移阈值（降低以减少过度切分）
    context_window_hours: int = 2         # 上下文窗口大小
    
    # 事件结构参数
    event_confidence_threshold: float = 0.5  # 降低事件置信度阈值
    
    # 关系建模参数
    similarity_threshold: float = 0.3     # 降低相似度阈值
    edge_confidence_threshold: float = 0.5  # 降低边置信度阈值
    max_candidates: int = 50
    
    # 层级抽象参数
    min_cluster_size: int = 2             # 最小聚类大小
    abstraction_threshold: float = 0.4    # 降低抽象阈值
    
    # 调试参数
    verbose: bool = False
    use_mock_llm: bool = True             # 使用模拟LLM（True用于演示）


cfg = MemoryOrganizationConfig(verbose=True, use_mock_llm=True)
print(f"Config initialized: {cfg}")

# %% [markdown]
# ## 2. 示例数据生成

# %%
# =============================================================================
# 2. 示例数据生成 / Sample Data Generation
# =============================================================================

def generate_sample_memories() -> List[MemoryNode]:
    """
    生成示例记忆数据
    模拟一个运维场景下的记忆序列
    """
    base_time = datetime(2025, 1, 7, 9, 0, 0)
    
    sample_data = [
        {
            "key": "网关延迟告警",
            "memory": "api-gateway服务的p99延迟从50ms飙升到500ms，触发告警",
            "tags": ["告警", "延迟", "api-gateway", "性能"],
            "background": "监控系统检测到api-gateway服务响应延迟异常升高",
            "offset_minutes": 0
        },
        {
            "key": "初步排查日志",
            "memory": "查看日志发现大量数据库连接超时错误",
            "tags": ["排查", "日志", "数据库", "连接超时"],
            "background": "工程师开始排查延迟问题的根因",
            "offset_minutes": 5
        },
        {
            "key": "数据库连接池分析",
            "memory": "检查数据库连接池配置，发现当前连接数100已达上限",
            "tags": ["分析", "连接池", "数据库", "配置"],
            "background": "定位到连接池配置可能是瓶颈",
            "offset_minutes": 15
        },
        {
            "key": "连接池扩容操作",
            "memory": "将数据库连接池从100扩容至200，并重启相关服务",
            "tags": ["操作", "扩容", "连接池", "重启"],
            "background": "执行扩容操作以缓解连接池压力",
            "offset_minutes": 25
        },
        {
            "key": "延迟恢复确认",
            "memory": "p99延迟恢复到正常水平45ms，告警自动解除",
            "tags": ["恢复", "验证", "延迟", "告警解除"],
            "background": "确认问题已解决，服务恢复正常",
            "offset_minutes": 35
        },
        {
            "key": "用户偏好记录",
            "memory": "用户偏好使用中文进行技术讨论",
            "tags": ["偏好", "语言", "中文"],
            "background": "记录用户的语言偏好设置",
            "offset_minutes": 100  # 时间间隔较大，应切分为新会话
        },
        {
            "key": "另一次延迟问题",
            "memory": "cache-service的响应延迟升高，怀疑是内存不足",
            "tags": ["告警", "延迟", "cache-service", "内存"],
            "background": "另一个服务出现类似的延迟问题",
            "offset_minutes": 200
        },
        {
            "key": "内存扩容处理",
            "memory": "为cache-service扩容内存从4GB到8GB，问题解决",
            "tags": ["操作", "扩容", "内存", "cache-service"],
            "background": "通过扩容内存解决了延迟问题",
            "offset_minutes": 220
        },
    ]
    
    memories = []
    for i, data in enumerate(sample_data):
        timestamp = base_time + timedelta(minutes=data["offset_minutes"])
        memory = MemoryNode(
            id=str(uuid.uuid4()),
            key=data["key"],
            memory=data["memory"],
            tags=data["tags"],
            type="event",
            confidence=0.95,
            created_at=timestamp.isoformat(),
            updated_at=timestamp.isoformat(),
            user_id="demo_user",
            session_id="default_session",
            background=data["background"],
            source_type="ops_log",
            embedding=None
        )
        memories.append(memory)
    
    return memories


# 生成示例数据
sample_memories = generate_sample_memories()
print(f"Generated {len(sample_memories)} sample memories:")
for m in sample_memories:
    print(f"  [{m.created_at[11:16]}] {m.key}")

# %% [markdown]
# ## 3. 模拟LLM与Embedding服务

# %%
# =============================================================================
# 3. 模拟LLM与Embedding服务 / Mock LLM & Embedding Services
# =============================================================================

class MockLLMService:
    """
    模拟LLM服务
    在实际使用中替换为真实的API调用（如OpenAI GPT、Claude等）
    """
    
    def __init__(self, cfg: MemoryOrganizationConfig):
        self.cfg = cfg
    
    def extract_event(self, text: str, memory_id: str, timestamp: str) -> EventUnit:
        """
        从文本中抽取事件要素
        实际实现应调用LLM API
        """
        # 模拟事件抽取逻辑
        keywords = {
            "告警": ("系统", "触发告警", None, "告警状态", "监控系统检测到异常"),
            "排查": ("工程师", "排查问题", "日志", "发现线索", "问题排查过程中"),
            "分析": ("工程师", "分析配置", "配置项", "定位问题", "深入分析后"),
            "扩容": ("运维", "执行扩容", "资源", "操作完成", "为解决问题"),
            "恢复": ("系统", "恢复正常", "服务", "问题解决", "处置后"),
            "偏好": ("用户", "设置偏好", "配置", None, "用户交互中"),
        }
        
        event_data = ("未知", "记录", None, None, None)
        for key, data in keywords.items():
            if key in text:
                event_data = data
                break
        
        return EventUnit(
            event_id=str(uuid.uuid4()),
            agent=event_data[0],
            action=event_data[1],
            object=event_data[2],
            outcome=event_data[3],
            context=event_data[4],
            timestamp=timestamp,
            source_memory_id=memory_id,
            confidence=0.85
        )
    
    def infer_relation(self, event_a: EventUnit, event_b: EventUnit) -> Tuple[RelationType, float]:
        """
        推断两个事件之间的关系
        实际实现应调用LLM API
        """
        # 简单的规则推断
        action_a = event_a.action.lower() if event_a.action else ""
        action_b = event_b.action.lower() if event_b.action else ""
        outcome_a = event_a.outcome.lower() if event_a.outcome else ""
        outcome_b = event_b.outcome.lower() if event_b.outcome else ""
        
        # 因果关系：A的结果导致B的发生
        if "告警" in action_a and ("排查" in action_b or "分析" in action_b or "记录" in action_b):
            return RelationType.CAUSES, 0.85
        if "扩容" in action_a and ("恢复" in action_b or "告警" in action_b):
            return RelationType.CAUSES, 0.9
        if "记录" in action_a and "扩容" in action_b:
            return RelationType.CAUSES, 0.7
        
        # 解决关系
        if "扩容" in action_a and "恢复" in outcome_b:
            return RelationType.RESOLVES, 0.9
        
        # 依赖关系
        if action_a and action_b:
            return RelationType.FOLLOWS, 0.6
        
        # 时间顺序关系
        return RelationType.FOLLOWS, 0.5
    
    def generate_abstraction(self, events: List[EventUnit]) -> Dict:
        """
        从事件簇中生成抽象模式
        实际实现应调用LLM API
        """
        # 分析事件共性
        actions = [e.action for e in events if e.action]
        outcomes = [e.outcome for e in events if e.outcome]
        
        has_scaling = any("扩容" in a for a in actions)
        has_recovery = any("恢复" in o or "解决" in o for o in outcomes)
        
        if has_scaling and has_recovery:
            return {
                "label": "资源扩容处置模式",
                "condition": "当服务延迟升高且资源使用率达到上限时",
                "solution": "对瓶颈资源（连接池、内存等）进行扩容",
                "verification": "监控延迟指标是否恢复正常",
                "confidence": 0.85
            }
        
        return {
            "label": "通用处置模式",
            "condition": "当系统出现异常时",
            "solution": "排查问题根因并采取相应措施",
            "verification": "确认系统恢复正常运行",
            "confidence": 0.7
        }


class MockEmbeddingService:
    """
    模拟Embedding服务
    在实际使用中替换为真实的API调用
    """
    
    def __init__(self, dim: int = 128):
        self.dim = dim
    
    def embed(self, text: str) -> List[float]:
        """
        生成文本的向量表示
        实际实现应调用Embedding API
        """
        # 使用文本的hash生成伪随机但确定性的向量
        hash_val = int(hashlib.md5(text.encode()).hexdigest(), 16)
        np.random.seed(hash_val % (2**32))
        embedding = np.random.randn(self.dim).tolist()
        # 归一化
        norm = np.linalg.norm(embedding)
        return [x / norm for x in embedding]
    
    def similarity(self, emb1: List[float], emb2: List[float]) -> float:
        """计算两个向量的余弦相似度"""
        dot = sum(a * b for a, b in zip(emb1, emb2))
        norm1 = np.sqrt(sum(a * a for a in emb1))
        norm2 = np.sqrt(sum(b * b for b in emb2))
        return dot / (norm1 * norm2) if norm1 > 0 and norm2 > 0 else 0.0


# 初始化服务
llm_service = MockLLMService(cfg)
embedding_service = MockEmbeddingService(dim=128)
print("LLM and Embedding services initialized (mock mode)")

# %% [markdown]
# ## 4. 模块一：时间结构构建

# %%
# =============================================================================
# 4. 模块一：时间结构构建 / Module 1: Temporal Structure
# =============================================================================

class TemporalStructureBuilder:
    """
    时间结构构建器
    负责将记忆按时间维度组织成会话和阶段
    
    核心功能：
    1. 时间规范化
    2. 会话/片段切分
    3. 阶段抽象
    4. 时间窗口上下文恢复
    """
    
    def __init__(self, cfg: MemoryOrganizationConfig):
        self.cfg = cfg
        self.sessions: List[Session] = []
        self.phases: List[Phase] = []
        self.memory_to_session: Dict[str, str] = {}
    
    def normalize_timestamp(self, timestamp: str) -> datetime:
        """
        时间规范化：将各种时间格式统一为datetime对象
        """
        try:
            # 处理ISO格式
            if 'T' in timestamp:
                return datetime.fromisoformat(timestamp.replace('Z', '+00:00').replace('+00:00', ''))
            # 处理其他格式...
            return datetime.fromisoformat(timestamp)
        except Exception as e:
            if self.cfg.verbose:
                print(f"Warning: Failed to parse timestamp {timestamp}: {e}")
            return datetime.now()
    
    def segment_into_sessions(
        self, 
        memories: List[MemoryNode],
        embedding_service: MockEmbeddingService = None
    ) -> List[Session]:
        """
        会话切分：基于时间间隔和主题漂移将记忆划分为会话
        
        Args:
            memories: 记忆节点列表
            embedding_service: embedding服务（用于主题漂移检测）
        
        Returns:
            会话列表
        """
        if not memories:
            return []
        
        # 按时间排序
        sorted_memories = sorted(memories, key=lambda m: self.normalize_timestamp(m.created_at))
        
        sessions = []
        current_session_memories = [sorted_memories[0]]
        
        for i in range(1, len(sorted_memories)):
            prev_memory = sorted_memories[i - 1]
            curr_memory = sorted_memories[i]
            
            prev_time = self.normalize_timestamp(prev_memory.created_at)
            curr_time = self.normalize_timestamp(curr_memory.created_at)
            
            # 条件1：时间间隔超过阈值
            time_gap = (curr_time - prev_time).total_seconds() / 60
            time_break = time_gap > self.cfg.time_threshold_minutes
            
            # 条件2：主题漂移检测（使用tag重叠度作为简化指标）
            topic_break = False
            if embedding_service:
                prev_emb = embedding_service.embed(prev_memory.memory)
                curr_emb = embedding_service.embed(curr_memory.memory)
                similarity = embedding_service.similarity(prev_emb, curr_emb)
                topic_break = similarity < self.cfg.topic_drift_threshold
            else:
                # 简化：使用tag重叠度
                prev_tags = set(prev_memory.tags)
                curr_tags = set(curr_memory.tags)
                overlap = len(prev_tags & curr_tags) / max(len(prev_tags | curr_tags), 1)
                topic_break = overlap < 0.2
            
            # 判断是否切分
            if time_break or topic_break:
                # 保存当前会话
                session = self._create_session(current_session_memories)
                sessions.append(session)
                current_session_memories = [curr_memory]
                
                if self.cfg.verbose:
                    reason = "time_gap" if time_break else "topic_drift"
                    print(f"  Session break at [{curr_memory.created_at[11:16]}]: {reason}")
            else:
                current_session_memories.append(curr_memory)
        
        # 保存最后一个会话
        if current_session_memories:
            session = self._create_session(current_session_memories)
            sessions.append(session)
        
        self.sessions = sessions
        
        # 建立记忆到会话的映射
        for session in sessions:
            for mem_id in session.memory_ids:
                self.memory_to_session[mem_id] = session.session_id
        
        return sessions
    
    def _create_session(self, memories: List[MemoryNode]) -> Session:
        """创建会话对象"""
        memory_ids = [m.id for m in memories]
        start_time = min(m.created_at for m in memories)
        end_time = max(m.created_at for m in memories)
        
        # 提取主题（使用最频繁的tag）
        all_tags = []
        for m in memories:
            all_tags.extend(m.tags)
        topic = max(set(all_tags), key=all_tags.count) if all_tags else None
        
        return Session(
            session_id=str(uuid.uuid4()),
            memory_ids=memory_ids,
            start_time=start_time,
            end_time=end_time,
            topic=topic
        )
    
    def build_phases(
        self, 
        sessions: List[Session],
        external_boundaries: List[datetime] = None
    ) -> List[Phase]:
        """
        阶段抽象：将会话归并为更高层次的阶段
        
        Args:
            sessions: 会话列表
            external_boundaries: 外部阶段边界（如版本发布日期）
        
        Returns:
            阶段列表
        """
        if not sessions:
            return []
        
        phases = []
        
        if external_boundaries:
            # 基于外部边界划分阶段
            phases = self._split_by_external_boundaries(sessions, external_boundaries)
        else:
            # 基于时间相近度自动划分
            phases = self._auto_cluster_phases(sessions)
        
        self.phases = phases
        return phases
    
    def _split_by_external_boundaries(
        self, 
        sessions: List[Session],
        boundaries: List[datetime]
    ) -> List[Phase]:
        """基于外部边界划分阶段"""
        sorted_boundaries = sorted(boundaries)
        phases = []
        current_phase_sessions = []
        boundary_idx = 0
        
        for session in sorted(sessions, key=lambda s: s.start_time):
            session_start = self.normalize_timestamp(session.start_time)
            
            while boundary_idx < len(sorted_boundaries) and session_start >= sorted_boundaries[boundary_idx]:
                if current_phase_sessions:
                    phase = self._create_phase(current_phase_sessions, f"Phase_{boundary_idx}")
                    phases.append(phase)
                    current_phase_sessions = []
                boundary_idx += 1
            
            current_phase_sessions.append(session)
        
        if current_phase_sessions:
            phase = self._create_phase(current_phase_sessions, f"Phase_{boundary_idx}")
            phases.append(phase)
        
        return phases
    
    def _auto_cluster_phases(self, sessions: List[Session]) -> List[Phase]:
        """自动聚类划分阶段（简化实现）"""
        # 简化：每天作为一个阶段
        day_groups = defaultdict(list)
        for session in sessions:
            day = self.normalize_timestamp(session.start_time).date()
            day_groups[day].append(session)
        
        phases = []
        for day, day_sessions in sorted(day_groups.items()):
            phase = self._create_phase(day_sessions, f"Phase_{day.isoformat()}")
            phases.append(phase)
        
        return phases
    
    def _create_phase(self, sessions: List[Session], label: str) -> Phase:
        """创建阶段对象"""
        session_ids = [s.session_id for s in sessions]
        start_time = min(s.start_time for s in sessions)
        end_time = max(s.end_time for s in sessions)
        
        # 生成阶段摘要
        topics = [s.topic for s in sessions if s.topic]
        summary = f"包含{len(sessions)}个会话，主要涉及：{', '.join(set(topics))}" if topics else None
        
        return Phase(
            phase_id=str(uuid.uuid4()),
            label=label,
            session_ids=session_ids,
            start_time=start_time,
            end_time=end_time,
            summary=summary
        )
    
    def retrieve_context_window(
        self,
        memories: List[MemoryNode],
        key_event_time: datetime,
        delta_hours: int = None
    ) -> List[MemoryNode]:
        """
        窗口式上下文恢复：获取关键事件前后时间窗口内的记忆
        
        Args:
            memories: 所有记忆节点
            key_event_time: 关键事件时间
            delta_hours: 时间窗口半径（小时）
        
        Returns:
            时间窗口内的记忆列表
        """
        if delta_hours is None:
            delta_hours = self.cfg.context_window_hours
        
        start = key_event_time - timedelta(hours=delta_hours)
        end = key_event_time + timedelta(hours=delta_hours)
        
        context_memories = []
        for m in memories:
            m_time = self.normalize_timestamp(m.created_at)
            if start <= m_time <= end:
                context_memories.append(m)
        
        return sorted(context_memories, key=lambda m: self.normalize_timestamp(m.created_at))
    
    def get_temporal_edges(self, memories: List[MemoryNode]) -> List[Edge]:
        """
        基于时间结构生成边
        
        Returns:
            时间关系边列表（FOLLOWS关系）
        """
        edges = []
        sorted_memories = sorted(memories, key=lambda m: self.normalize_timestamp(m.created_at))
        
        for i in range(len(sorted_memories) - 1):
            curr = sorted_memories[i]
            next_mem = sorted_memories[i + 1]
            
            # 检查时间间隔
            curr_time = self.normalize_timestamp(curr.created_at)
            next_time = self.normalize_timestamp(next_mem.created_at)
            time_gap_minutes = (next_time - curr_time).total_seconds() / 60
            
            # 根据时间间隔设置权重
            same_session = self.memory_to_session.get(curr.id) == self.memory_to_session.get(next_mem.id)
            
            # 在时间阈值内都建立边，同会话权重更高
            if time_gap_minutes <= self.cfg.time_threshold_minutes * 2:
                weight = 0.9 if same_session else 0.6
                edge = Edge(
                    source_id=curr.id,
                    target_id=next_mem.id,
                    relation_type=RelationType.FOLLOWS,
                    weight=weight,
                    metadata={
                        "type": "temporal", 
                        "same_session": same_session,
                        "time_gap_minutes": time_gap_minutes
                    }
                )
                edges.append(edge)
        
        return edges


# %%
# 运行时间结构模块 / Run Temporal Structure Module
print("=" * 60)
print("Module 1: Temporal Structure Building")
print("=" * 60)

temporal_builder = TemporalStructureBuilder(cfg)

# 会话切分
sessions = temporal_builder.segment_into_sessions(sample_memories, embedding_service)
print(f"\nSegmented into {len(sessions)} sessions:")
for s in sessions:
    print(f"  Session [{s.start_time[11:16]} - {s.end_time[11:16]}]: {len(s.memory_ids)} memories, topic: {s.topic}")

# 阶段构建
phases = temporal_builder.build_phases(sessions)
print(f"\nBuilt {len(phases)} phases:")
for p in phases:
    print(f"  {p.label}: {len(p.session_ids)} sessions, {p.summary}")

# 时间边生成
temporal_edges = temporal_builder.get_temporal_edges(sample_memories)
print(f"\nGenerated {len(temporal_edges)} temporal edges (FOLLOWS relations)")

# 上下文窗口示例
key_time = temporal_builder.normalize_timestamp(sample_memories[3].created_at)
context = temporal_builder.retrieve_context_window(sample_memories, key_time, delta_hours=1)
print(f"\nContext window around [{sample_memories[3].key}]:")
for m in context:
    print(f"  [{m.created_at[11:16]}] {m.key}")

# %% [markdown]
# ## 5. 模块二：事件结构构建

# %%
# =============================================================================
# 5. 模块二：事件结构构建 / Module 2: Event Structure
# =============================================================================

class EventStructureBuilder:
    """
    事件结构构建器
    负责从记忆中抽取事件要素并建立事件间关系
    
    核心功能：
    1. 事件抽取（主体、动作、对象、结果、上下文）
    2. 事件关系推断（因果、依赖、解决）
    3. 事件链构建
    """
    
    def __init__(self, cfg: MemoryOrganizationConfig, llm_service: MockLLMService):
        self.cfg = cfg
        self.llm = llm_service
        self.events: Dict[str, EventUnit] = {}  # event_id -> EventUnit
        self.memory_to_event: Dict[str, str] = {}  # memory_id -> event_id
    
    def extract_events(self, memories: List[MemoryNode]) -> List[EventUnit]:
        """
        从记忆中抽取事件要素
        
        Args:
            memories: 记忆节点列表
        
        Returns:
            事件单元列表
        """
        events = []
        
        for memory in memories:
            event = self.llm.extract_event(
                text=memory.memory,
                memory_id=memory.id,
                timestamp=memory.created_at
            )
            
            events.append(event)
            self.events[event.event_id] = event
            self.memory_to_event[memory.id] = event.event_id
            
            if self.cfg.verbose:
                print(f"  Extracted event: {event.agent} -> {event.action} -> {event.outcome}")
        
        return events
    
    def infer_event_relations(
        self, 
        events: List[EventUnit],
        temporal_edges: List[Edge] = None
    ) -> List[Edge]:
        """
        推断事件之间的关系
        
        Args:
            events: 事件列表
            temporal_edges: 时间边（可选，用于约束关系方向）
        
        Returns:
            事件关系边列表
        """
        edges = []
        
        # 建立时间顺序索引
        sorted_events = sorted(events, key=lambda e: e.timestamp)
        
        # 使用滑动窗口推断关系
        for i, event_a in enumerate(sorted_events):
            # 只看后续几个事件
            for j in range(i + 1, min(i + 5, len(sorted_events))):
                event_b = sorted_events[j]
                
                # 跳过置信度低的事件
                if event_a.confidence < self.cfg.event_confidence_threshold:
                    continue
                if event_b.confidence < self.cfg.event_confidence_threshold:
                    continue
                
                # 推断关系
                relation_type, confidence = self.llm.infer_relation(event_a, event_b)
                
                if confidence >= self.cfg.edge_confidence_threshold:
                    edge = Edge(
                        source_id=event_a.source_memory_id,
                        target_id=event_b.source_memory_id,
                        relation_type=relation_type,
                        weight=confidence,
                        metadata={
                            "type": "event",
                            "event_a_id": event_a.event_id,
                            "event_b_id": event_b.event_id,
                            "action_a": event_a.action,
                            "action_b": event_b.action
                        }
                    )
                    edges.append(edge)
                    
                    if self.cfg.verbose:
                        print(f"  Relation: {event_a.action} --[{relation_type.value}]--> {event_b.action}")
        
        return edges
    
    def build_event_chains(self, edges: List[Edge]) -> List[List[str]]:
        """
        构建事件链：将相连的事件组织成链条
        
        Args:
            edges: 事件关系边
        
        Returns:
            事件链列表（每条链是记忆ID的列表）
        """
        # 构建邻接表
        graph = defaultdict(list)
        in_degree = defaultdict(int)
        nodes = set()
        
        for edge in edges:
            if edge.relation_type in [RelationType.CAUSES, RelationType.FOLLOWS, RelationType.RESOLVES]:
                graph[edge.source_id].append(edge.target_id)
                in_degree[edge.target_id] += 1
                nodes.add(edge.source_id)
                nodes.add(edge.target_id)
        
        # 找出所有起点（入度为0的节点）
        start_nodes = [n for n in nodes if in_degree[n] == 0]
        
        # DFS构建链
        chains = []
        visited = set()
        
        def dfs(node: str, chain: List[str]):
            chain.append(node)
            visited.add(node)
            
            if not graph[node]:
                # 到达终点
                if len(chain) > 1:
                    chains.append(chain.copy())
            else:
                for next_node in graph[node]:
                    if next_node not in visited:
                        dfs(next_node, chain)
            
            chain.pop()
            visited.discard(node)
        
        for start in start_nodes:
            dfs(start, [])
        
        return chains
    
    def get_event_structure(self, memory_id: str) -> Optional[EventUnit]:
        """获取记忆对应的事件结构"""
        event_id = self.memory_to_event.get(memory_id)
        return self.events.get(event_id) if event_id else None


# %%
# 运行事件结构模块 / Run Event Structure Module
print("=" * 60)
print("Module 2: Event Structure Building")
print("=" * 60)

event_builder = EventStructureBuilder(cfg, llm_service)

# 事件抽取
print("\nExtracting events from memories:")
events = event_builder.extract_events(sample_memories)
print(f"Extracted {len(events)} events")

# 事件关系推断
print("\nInferring event relations:")
event_edges = event_builder.infer_event_relations(events, temporal_edges)
print(f"Inferred {len(event_edges)} event relations")

# 事件链构建
chains = event_builder.build_event_chains(event_edges)
print(f"\nBuilt {len(chains)} event chains:")

# 将记忆ID映射回关键词以便展示
id_to_key = {m.id: m.key for m in sample_memories}
for i, chain in enumerate(chains):
    chain_str = " -> ".join([id_to_key.get(mid, mid[:8]) for mid in chain])
    print(f"  Chain {i+1}: {chain_str}")

# %% [markdown]
# ## 6. 模块三：语义结构与关系建模

# %%
# =============================================================================
# 6. 模块三：语义结构与关系建模 / Module 3: Semantic Structure & Relation Modeling
# =============================================================================

@dataclass
class StructuredMemory:
    """
    结构化记忆：在原始记忆基础上增加语义属性
    """
    original: MemoryNode
    topic: Optional[str] = None
    topic_confidence: float = 0.0
    entities: List[Dict] = field(default_factory=list)
    slots: Dict[str, str] = field(default_factory=dict)
    embedding: List[float] = field(default_factory=list)


class SemanticStructureBuilder:
    """
    语义结构构建器
    负责对记忆进行语义结构化和关系建模
    
    核心功能：
    1. 语义结构化（topic、entities、slots）
    2. 候选边生成（规则、相似度、聚类）
    3. 关系判别
    """
    
    def __init__(
        self, 
        cfg: MemoryOrganizationConfig,
        embedding_service: MockEmbeddingService
    ):
        self.cfg = cfg
        self.embedding = embedding_service
        self.structured_memories: Dict[str, StructuredMemory] = {}
    
    def structure_memory(self, memory: MemoryNode) -> StructuredMemory:
        """
        对单条记忆进行语义结构化
        
        Args:
            memory: 原始记忆节点
        
        Returns:
            结构化记忆
        """
        # 1. 生成embedding
        emb = self.embedding.embed(memory.memory)
        
        # 2. 提取topic（简化：使用最相关的tag）
        topic = memory.tags[0] if memory.tags else None
        topic_conf = 0.8 if topic else 0.0
        
        # 3. 提取entities（简化实现）
        entities = self._extract_entities(memory.memory)
        
        # 4. 提取slots（简化实现）
        slots = self._extract_slots(memory)
        
        structured = StructuredMemory(
            original=memory,
            topic=topic,
            topic_confidence=topic_conf,
            entities=entities,
            slots=slots,
            embedding=emb
        )
        
        self.structured_memories[memory.id] = structured
        return structured
    
    def _extract_entities(self, text: str) -> List[Dict]:
        """抽取实体（简化实现）"""
        entities = []
        
        # 服务实体
        services = ["api-gateway", "cache-service", "database"]
        for svc in services:
            if svc in text.lower():
                entities.append({
                    "type": "Service",
                    "id": svc,
                    "name": svc
                })
        
        # 指标实体
        metrics = ["p99", "延迟", "响应时间", "连接数"]
        for metric in metrics:
            if metric in text:
                entities.append({
                    "type": "Metric",
                    "id": metric,
                    "name": metric
                })
        
        # 资源实体
        resources = ["连接池", "内存", "CPU"]
        for res in resources:
            if res in text:
                entities.append({
                    "type": "Resource",
                    "id": res,
                    "name": res
                })
        
        return entities
    
    def _extract_slots(self, memory: MemoryNode) -> Dict[str, str]:
        """抽取slots（简化实现）"""
        slots = {}
        text = memory.memory
        
        # 服务slot
        services = ["api-gateway", "cache-service"]
        for svc in services:
            if svc in text.lower():
                slots["service"] = svc
                break
        
        # 指标slot
        if "p99" in text or "延迟" in text:
            slots["metric"] = "latency"
        
        # 动作slot
        if "扩容" in text:
            slots["action_type"] = "scaling"
        elif "排查" in text:
            slots["action_type"] = "investigation"
        elif "告警" in text:
            slots["action_type"] = "alerting"
        
        # 数值变化slot
        import re
        numbers = re.findall(r'\d+', text)
        if len(numbers) >= 2:
            slots["value_change"] = f"{numbers[0]}->{numbers[1]}"
        
        return slots
    
    def generate_candidate_edges(
        self, 
        new_memory_id: str,
        all_structured: Dict[str, StructuredMemory]
    ) -> List[str]:
        """
        生成候选边（三路来源融合）
        
        Args:
            new_memory_id: 新记忆ID
            all_structured: 所有结构化记忆
        
        Returns:
            候选记忆ID列表
        """
        if new_memory_id not in all_structured:
            return []
        
        new_mem = all_structured[new_memory_id]
        candidates = set()
        
        # 路径1：规则过滤（topic/slot匹配）
        for mem_id, mem in all_structured.items():
            if mem_id == new_memory_id:
                continue
            
            # 同topic
            if new_mem.topic and mem.topic == new_mem.topic:
                candidates.add(mem_id)
            
            # 同service slot
            if new_mem.slots.get("service") and new_mem.slots.get("service") == mem.slots.get("service"):
                candidates.add(mem_id)
        
        # 路径2：相似度检索Top-K
        similarities = []
        for mem_id, mem in all_structured.items():
            if mem_id == new_memory_id:
                continue
            sim = self.embedding.similarity(new_mem.embedding, mem.embedding)
            similarities.append((mem_id, sim))
        
        similarities.sort(key=lambda x: x[1], reverse=True)
        for mem_id, sim in similarities[:10]:  # Top-10
            if sim >= self.cfg.similarity_threshold:
                candidates.add(mem_id)
        
        # 路径3：实体共现
        new_entity_ids = {e["id"] for e in new_mem.entities}
        for mem_id, mem in all_structured.items():
            if mem_id == new_memory_id:
                continue
            mem_entity_ids = {e["id"] for e in mem.entities}
            if new_entity_ids & mem_entity_ids:  # 有交集
                candidates.add(mem_id)
        
        # 控制候选集规模
        return list(candidates)[:self.cfg.max_candidates]
    
    def classify_relation(
        self,
        mem_a: StructuredMemory,
        mem_b: StructuredMemory
    ) -> Tuple[Optional[RelationType], float]:
        """
        判别两个记忆之间的关系类型
        
        Returns:
            (关系类型, 置信度) 或 (None, 0) 如果无关系
        """
        # 1. 检查slot匹配度
        slot_overlap = 0
        for k, v in mem_a.slots.items():
            if mem_b.slots.get(k) == v:
                slot_overlap += 1
        
        # 2. 检查实体重叠
        entities_a = {e["id"] for e in mem_a.entities}
        entities_b = {e["id"] for e in mem_b.entities}
        entity_overlap = len(entities_a & entities_b)
        
        # 3. 计算语义相似度
        semantic_sim = self.embedding.similarity(mem_a.embedding, mem_b.embedding)
        
        # 综合判断
        if semantic_sim >= 0.8 and slot_overlap >= 2:
            return RelationType.SAME_TOPIC, 0.9
        
        if slot_overlap >= 1 and entity_overlap >= 1:
            return RelationType.RELATED_TO, 0.75
        
        if semantic_sim >= 0.6:
            return RelationType.RELATED_TO, semantic_sim
        
        return None, 0.0
    
    def build_semantic_edges(
        self,
        memories: List[MemoryNode]
    ) -> List[Edge]:
        """
        构建语义关系边
        
        Args:
            memories: 记忆列表
        
        Returns:
            语义关系边列表
        """
        # 1. 结构化所有记忆
        for mem in memories:
            if mem.id not in self.structured_memories:
                self.structure_memory(mem)
        
        edges = []
        processed_pairs = set()
        
        # 2. 为每个记忆生成候选边并判别
        for mem_id in self.structured_memories:
            candidates = self.generate_candidate_edges(mem_id, self.structured_memories)
            mem_a = self.structured_memories[mem_id]
            
            for cand_id in candidates:
                pair_key = tuple(sorted([mem_id, cand_id]))
                if pair_key in processed_pairs:
                    continue
                processed_pairs.add(pair_key)
                
                mem_b = self.structured_memories[cand_id]
                relation_type, confidence = self.classify_relation(mem_a, mem_b)
                
                if relation_type and confidence >= self.cfg.edge_confidence_threshold:
                    edge = Edge(
                        source_id=mem_id,
                        target_id=cand_id,
                        relation_type=relation_type,
                        weight=confidence,
                        metadata={
                            "type": "semantic",
                            "slot_match": mem_a.slots == mem_b.slots,
                            "same_topic": mem_a.topic == mem_b.topic
                        }
                    )
                    edges.append(edge)
        
        return edges


# %%
# 运行语义结构模块 / Run Semantic Structure Module
print("=" * 60)
print("Module 3: Semantic Structure & Relation Modeling")
print("=" * 60)

semantic_builder = SemanticStructureBuilder(cfg, embedding_service)

# 结构化所有记忆
print("\nStructuring memories:")
for mem in sample_memories:
    structured = semantic_builder.structure_memory(mem)
    print(f"  [{mem.key}]")
    print(f"    topic: {structured.topic}, slots: {structured.slots}")
    print(f"    entities: {[e['id'] for e in structured.entities]}")

# 构建语义边
semantic_edges = semantic_builder.build_semantic_edges(sample_memories)
print(f"\nBuilt {len(semantic_edges)} semantic edges:")
for edge in semantic_edges:
    src_key = id_to_key.get(edge.source_id, edge.source_id[:8])
    tgt_key = id_to_key.get(edge.target_id, edge.target_id[:8])
    print(f"  {src_key} --[{edge.relation_type.value}:{edge.weight:.2f}]--> {tgt_key}")

# %% [markdown]
# ## 7. 模块四：层级化与抽象

# %%
# =============================================================================
# 7. 模块四：层级化与抽象 / Module 4: Hierarchical Abstraction
# =============================================================================

@dataclass
class IndexNode:
    """索引树节点"""
    node_id: str
    label: str
    level: int
    parent_id: Optional[str]
    children_ids: List[str] = field(default_factory=list)
    memory_ids: List[str] = field(default_factory=list)
    summary: Optional[str] = None
    embedding: Optional[List[float]] = None


class HierarchicalAbstractionBuilder:
    """
    层级化与抽象构建器
    负责构建索引树和生成抽象节点
    
    核心功能：
    1. 索引树构建
    2. 自动聚合
    3. 模式归纳
    4. 抽象固化
    """
    
    def __init__(
        self, 
        cfg: MemoryOrganizationConfig,
        llm_service: MockLLMService,
        embedding_service: MockEmbeddingService
    ):
        self.cfg = cfg
        self.llm = llm_service
        self.embedding = embedding_service
        self.index_nodes: Dict[str, IndexNode] = {}
        self.abstractions: Dict[str, AbstractionNode] = {}
        self.root_id: Optional[str] = None
    
    def build_index_tree(
        self,
        structured_memories: Dict[str, StructuredMemory]
    ) -> str:
        """
        构建索引树
        
        Args:
            structured_memories: 结构化记忆字典
        
        Returns:
            根节点ID
        """
        # 创建根节点
        root_id = str(uuid.uuid4())
        root = IndexNode(
            node_id=root_id,
            label="Root",
            level=0,
            parent_id=None,
            summary="所有记忆的根节点"
        )
        self.index_nodes[root_id] = root
        self.root_id = root_id
        
        # 按topic分组
        topic_groups = defaultdict(list)
        for mem_id, mem in structured_memories.items():
            topic = mem.topic or "uncategorized"
            topic_groups[topic].append(mem_id)
        
        # 为每个topic创建中间节点
        for topic, mem_ids in topic_groups.items():
            topic_node_id = str(uuid.uuid4())
            topic_node = IndexNode(
                node_id=topic_node_id,
                label=f"Topic:{topic}",
                level=1,
                parent_id=root_id,
                memory_ids=mem_ids,
                summary=f"包含{len(mem_ids)}条关于'{topic}'的记忆"
            )
            self.index_nodes[topic_node_id] = topic_node
            root.children_ids.append(topic_node_id)
            
            if self.cfg.verbose:
                print(f"  Created topic node: {topic} with {len(mem_ids)} memories")
        
        return root_id
    
    def aggregate_similar_memories(
        self,
        structured_memories: Dict[str, StructuredMemory],
        min_cluster_size: int = None
    ) -> List[List[str]]:
        """
        聚合相似记忆
        
        Args:
            structured_memories: 结构化记忆
            min_cluster_size: 最小聚类大小
        
        Returns:
            聚类结果（记忆ID列表的列表）
        """
        if min_cluster_size is None:
            min_cluster_size = self.cfg.min_cluster_size
        
        # 简化的聚类：基于slot相似度
        clusters = []
        used = set()
        
        mem_list = list(structured_memories.items())
        
        for i, (mem_id, mem) in enumerate(mem_list):
            if mem_id in used:
                continue
            
            cluster = [mem_id]
            used.add(mem_id)
            
            for j in range(i + 1, len(mem_list)):
                other_id, other = mem_list[j]
                if other_id in used:
                    continue
                
                # 检查slot相似度
                common_slots = set(mem.slots.keys()) & set(other.slots.keys())
                if not common_slots:
                    continue
                
                match_count = sum(1 for k in common_slots if mem.slots[k] == other.slots[k])
                if match_count >= 1:
                    # 检查语义相似度
                    sim = self.embedding.similarity(mem.embedding, other.embedding)
                    if sim >= self.cfg.abstraction_threshold:
                        cluster.append(other_id)
                        used.add(other_id)
            
            if len(cluster) >= min_cluster_size:
                clusters.append(cluster)
        
        return clusters
    
    def generate_abstraction(
        self,
        cluster_memory_ids: List[str],
        structured_memories: Dict[str, StructuredMemory],
        events: List[EventUnit]
    ) -> AbstractionNode:
        """
        从案例簇生成抽象模式
        
        Args:
            cluster_memory_ids: 簇中的记忆ID列表
            structured_memories: 结构化记忆
            events: 相关事件列表
        
        Returns:
            抽象节点
        """
        # 获取相关事件
        cluster_events = [e for e in events if e.source_memory_id in cluster_memory_ids]
        
        # 调用LLM生成抽象
        abstraction_data = self.llm.generate_abstraction(cluster_events)
        
        abstraction = AbstractionNode(
            node_id=str(uuid.uuid4()),
            label=abstraction_data["label"],
            condition=abstraction_data["condition"],
            solution=abstraction_data["solution"],
            verification=abstraction_data["verification"],
            support_ids=cluster_memory_ids,
            confidence=abstraction_data["confidence"]
        )
        
        self.abstractions[abstraction.node_id] = abstraction
        return abstraction
    
    def attach_abstraction_to_tree(
        self,
        abstraction: AbstractionNode,
        parent_node_id: str = None
    ):
        """
        将抽象节点附加到索引树
        """
        if parent_node_id is None:
            parent_node_id = self.root_id
        
        parent = self.index_nodes.get(parent_node_id)
        if not parent:
            return
        
        # 创建抽象的索引节点
        abs_index_node = IndexNode(
            node_id=abstraction.node_id,
            label=f"Pattern:{abstraction.label}",
            level=parent.level + 1,
            parent_id=parent_node_id,
            memory_ids=abstraction.support_ids,
            summary=f"抽象模式：{abstraction.condition}"
        )
        
        self.index_nodes[abstraction.node_id] = abs_index_node
        parent.children_ids.append(abstraction.node_id)
    
    def get_abstraction_edges(self) -> List[Edge]:
        """
        生成抽象关系边（SUPPORTS关系）
        """
        edges = []
        
        for abs_id, abstraction in self.abstractions.items():
            for support_id in abstraction.support_ids:
                edge = Edge(
                    source_id=abs_id,
                    target_id=support_id,
                    relation_type=RelationType.CONTAINS,
                    weight=abstraction.confidence,
                    metadata={
                        "type": "abstraction",
                        "pattern_label": abstraction.label
                    }
                )
                edges.append(edge)
        
        return edges
    
    def print_tree(self, node_id: str = None, indent: int = 0):
        """打印索引树结构"""
        if node_id is None:
            node_id = self.root_id
        
        if node_id not in self.index_nodes:
            return
        
        node = self.index_nodes[node_id]
        prefix = "  " * indent
        print(f"{prefix}[{node.label}] - {len(node.memory_ids)} memories")
        
        for child_id in node.children_ids:
            self.print_tree(child_id, indent + 1)


# %%
# 运行层级抽象模块 / Run Hierarchical Abstraction Module
print("=" * 60)
print("Module 4: Hierarchical Abstraction")
print("=" * 60)

hierarchy_builder = HierarchicalAbstractionBuilder(cfg, llm_service, embedding_service)

# 构建索引树
print("\nBuilding index tree:")
root_id = hierarchy_builder.build_index_tree(semantic_builder.structured_memories)

# 聚合相似记忆
print("\nAggregating similar memories:")
clusters = hierarchy_builder.aggregate_similar_memories(
    semantic_builder.structured_memories,
    min_cluster_size=2  # 降低阈值以便演示
)
print(f"Found {len(clusters)} clusters:")
for i, cluster in enumerate(clusters):
    cluster_keys = [id_to_key.get(mid, mid[:8]) for mid in cluster]
    print(f"  Cluster {i+1}: {cluster_keys}")

# 生成抽象
print("\nGenerating abstractions:")
for cluster in clusters:
    abstraction = hierarchy_builder.generate_abstraction(
        cluster,
        semantic_builder.structured_memories,
        events
    )
    hierarchy_builder.attach_abstraction_to_tree(abstraction)
    print(f"  Abstraction: {abstraction.label}")
    print(f"    Condition: {abstraction.condition}")
    print(f"    Solution: {abstraction.solution}")

# 打印索引树
print("\nIndex Tree Structure:")
hierarchy_builder.print_tree()

# 抽象边
abstraction_edges = hierarchy_builder.get_abstraction_edges()
print(f"\nGenerated {len(abstraction_edges)} abstraction edges")

# %% [markdown]
# ## 8. 记忆图谱整合

# %%
# =============================================================================
# 8. 记忆图谱整合 / Memory Graph Integration
# =============================================================================

@dataclass
class MemoryGraph:
    """
    记忆图谱：整合所有组织结构的完整图
    """
    nodes: Dict[str, MemoryNode]
    edges: List[Edge]
    sessions: List[Session]
    phases: List[Phase]
    abstractions: Dict[str, AbstractionNode]
    index_tree_root: Optional[str]
    
    def get_node_count(self) -> int:
        return len(self.nodes)
    
    def get_edge_count(self) -> int:
        return len(self.edges)
    
    def get_edges_by_type(self, relation_type: RelationType) -> List[Edge]:
        return [e for e in self.edges if e.relation_type == relation_type]
    
    def get_neighbors(self, node_id: str) -> List[str]:
        neighbors = set()
        for edge in self.edges:
            if edge.source_id == node_id:
                neighbors.add(edge.target_id)
            elif edge.target_id == node_id:
                neighbors.add(edge.source_id)
        return list(neighbors)
    
    def to_dict(self) -> Dict:
        return {
            "nodes": {k: v.to_dict() for k, v in self.nodes.items()},
            "edges": [asdict(e) for e in self.edges],
            "sessions": [asdict(s) for s in self.sessions],
            "phases": [asdict(p) for p in self.phases],
            "abstractions": {k: asdict(v) for k, v in self.abstractions.items()},
            "index_tree_root": self.index_tree_root,
            "stats": {
                "node_count": self.get_node_count(),
                "edge_count": self.get_edge_count()
            }
        }


class MemoryOrganizer:
    """
    记忆组织器：整合所有模块的主入口
    """
    
    def __init__(self, cfg: MemoryOrganizationConfig):
        self.cfg = cfg
        self.llm = MockLLMService(cfg)
        self.embedding = MockEmbeddingService()
        
        self.temporal_builder = TemporalStructureBuilder(cfg)
        self.event_builder = EventStructureBuilder(cfg, self.llm)
        self.semantic_builder = SemanticStructureBuilder(cfg, self.embedding)
        self.hierarchy_builder = HierarchicalAbstractionBuilder(cfg, self.llm, self.embedding)
    
    def organize(
        self,
        memories: List[MemoryNode],
        use_temporal: bool = True,
        use_event: bool = True,
        use_semantic: bool = True,
        use_hierarchy: bool = True
    ) -> MemoryGraph:
        """
        执行完整的记忆组织流程
        
        Args:
            memories: 原始记忆列表
            use_temporal: 是否使用时间结构
            use_event: 是否使用事件结构
            use_semantic: 是否使用语义结构
            use_hierarchy: 是否使用层级抽象
        
        Returns:
            组织好的记忆图谱
        """
        all_edges = []
        sessions = []
        phases = []
        abstractions = {}
        index_tree_root = None
        events = []
        
        # 1. 时间结构
        if use_temporal:
            if self.cfg.verbose:
                print("\n[Step 1] Building temporal structure...")
            sessions = self.temporal_builder.segment_into_sessions(memories, self.embedding)
            phases = self.temporal_builder.build_phases(sessions)
            temporal_edges = self.temporal_builder.get_temporal_edges(memories)
            all_edges.extend(temporal_edges)
            if self.cfg.verbose:
                print(f"  Sessions: {len(sessions)}, Phases: {len(phases)}, Edges: {len(temporal_edges)}")
        
        # 2. 事件结构
        if use_event:
            if self.cfg.verbose:
                print("\n[Step 2] Building event structure...")
            events = self.event_builder.extract_events(memories)
            event_edges = self.event_builder.infer_event_relations(events)
            all_edges.extend(event_edges)
            if self.cfg.verbose:
                print(f"  Events: {len(events)}, Edges: {len(event_edges)}")
        
        # 3. 语义结构
        if use_semantic:
            if self.cfg.verbose:
                print("\n[Step 3] Building semantic structure...")
            semantic_edges = self.semantic_builder.build_semantic_edges(memories)
            all_edges.extend(semantic_edges)
            if self.cfg.verbose:
                print(f"  Structured: {len(self.semantic_builder.structured_memories)}, Edges: {len(semantic_edges)}")
        
        # 4. 层级抽象
        if use_hierarchy and use_semantic:
            if self.cfg.verbose:
                print("\n[Step 4] Building hierarchical abstraction...")
            index_tree_root = self.hierarchy_builder.build_index_tree(
                self.semantic_builder.structured_memories
            )
            clusters = self.hierarchy_builder.aggregate_similar_memories(
                self.semantic_builder.structured_memories,
                min_cluster_size=2
            )
            for cluster in clusters:
                abstraction = self.hierarchy_builder.generate_abstraction(
                    cluster,
                    self.semantic_builder.structured_memories,
                    events
                )
                self.hierarchy_builder.attach_abstraction_to_tree(abstraction)
            
            abstraction_edges = self.hierarchy_builder.get_abstraction_edges()
            all_edges.extend(abstraction_edges)
            abstractions = self.hierarchy_builder.abstractions
            if self.cfg.verbose:
                print(f"  Abstractions: {len(abstractions)}, Edges: {len(abstraction_edges)}")
        
        # 去重边
        unique_edges = self._deduplicate_edges(all_edges)
        
        # 构建图谱
        nodes = {m.id: m for m in memories}
        
        graph = MemoryGraph(
            nodes=nodes,
            edges=unique_edges,
            sessions=sessions,
            phases=phases,
            abstractions=abstractions,
            index_tree_root=index_tree_root
        )
        
        return graph
    
    def _deduplicate_edges(self, edges: List[Edge]) -> List[Edge]:
        """去重边（保留权重最高的）"""
        edge_map = {}
        for edge in edges:
            key = (edge.source_id, edge.target_id, edge.relation_type)
            if key not in edge_map or edge.weight > edge_map[key].weight:
                edge_map[key] = edge
        return list(edge_map.values())


# %%
# 运行完整组织流程 / Run Full Organization Pipeline
print("=" * 60)
print("Full Memory Organization Pipeline")
print("=" * 60)

organizer = MemoryOrganizer(cfg)
memory_graph = organizer.organize(
    sample_memories,
    use_temporal=True,
    use_event=True,
    use_semantic=True,
    use_hierarchy=True
)

print("\n" + "=" * 60)
print("Memory Graph Summary")
print("=" * 60)
print(f"Total nodes: {memory_graph.get_node_count()}")
print(f"Total edges: {memory_graph.get_edge_count()}")
print(f"Sessions: {len(memory_graph.sessions)}")
print(f"Phases: {len(memory_graph.phases)}")
print(f"Abstractions: {len(memory_graph.abstractions)}")

print("\nEdge distribution by type:")
for rel_type in RelationType:
    count = len(memory_graph.get_edges_by_type(rel_type))
    if count > 0:
        print(f"  {rel_type.value}: {count}")

# %% [markdown]
# ## 9. 基础校验

# %%
# =============================================================================
# 9. 基础校验 / Validation & Sanity Checks
# =============================================================================

def test_memory_node():
    """测试记忆节点基本功能"""
    node = MemoryNode(
        id="test-1",
        key="测试",
        memory="这是测试记忆",
        tags=["test"],
        type="fact",
        confidence=0.9,
        created_at="2025-01-07T10:00:00",
        updated_at="2025-01-07T10:00:00",
        user_id="user1",
        session_id="session1"
    )
    assert node.id == "test-1"
    assert len(node.tags) == 1
    d = node.to_dict()
    assert isinstance(d, dict)
    node2 = MemoryNode.from_dict(d)
    assert node2.id == node.id
    print("✓ MemoryNode tests passed")


def test_temporal_structure():
    """测试时间结构"""
    cfg = MemoryOrganizationConfig(verbose=False)
    builder = TemporalStructureBuilder(cfg)
    
    # 测试时间规范化
    ts = builder.normalize_timestamp("2025-01-07T10:00:00")
    assert ts.year == 2025
    
    # 测试会话切分
    sessions = builder.segment_into_sessions(sample_memories)
    assert len(sessions) >= 1
    assert all(len(s.memory_ids) > 0 for s in sessions)
    print("✓ TemporalStructure tests passed")


def test_event_structure():
    """测试事件结构"""
    cfg = MemoryOrganizationConfig(verbose=False)
    llm = MockLLMService(cfg)
    builder = EventStructureBuilder(cfg, llm)
    
    events = builder.extract_events(sample_memories[:3])
    assert len(events) == 3
    assert all(e.action for e in events)
    print("✓ EventStructure tests passed")


def test_semantic_structure():
    """测试语义结构"""
    cfg = MemoryOrganizationConfig(verbose=False)
    emb = MockEmbeddingService()
    builder = SemanticStructureBuilder(cfg, emb)
    
    structured = builder.structure_memory(sample_memories[0])
    assert structured.topic is not None
    assert len(structured.embedding) > 0
    print("✓ SemanticStructure tests passed")


def test_full_pipeline():
    """测试完整流程"""
    cfg = MemoryOrganizationConfig(verbose=False)
    organizer = MemoryOrganizer(cfg)
    graph = organizer.organize(sample_memories)
    
    assert graph.get_node_count() > 0
    assert graph.get_edge_count() > 0
    print("✓ Full pipeline tests passed")


# 运行所有测试
print("Running validation tests...")
test_memory_node()
test_temporal_structure()
test_event_structure()
test_semantic_structure()
test_full_pipeline()
print("\n" + "=" * 60)
print("All tests passed!")
print("=" * 60)

# %% [markdown]
# ## 10. 使用示例：推理查询

# %%
# =============================================================================
# 10. 使用示例：推理查询 / Example: Inference Query
# =============================================================================

def query_related_memories(
    graph: MemoryGraph,
    query_memory_id: str,
    max_hops: int = 2
) -> List[Tuple[str, int, str]]:
    """
    查询与指定记忆相关的所有记忆（多跳）
    
    Args:
        graph: 记忆图谱
        query_memory_id: 查询记忆ID
        max_hops: 最大跳数
    
    Returns:
        (记忆ID, 跳数, 关系路径) 列表
    """
    results = []
    visited = {query_memory_id}
    queue = [(query_memory_id, 0, "")]
    
    while queue:
        current_id, hops, path = queue.pop(0)
        
        if hops > 0:
            results.append((current_id, hops, path))
        
        if hops >= max_hops:
            continue
        
        for edge in graph.edges:
            next_id = None
            direction = ""
            
            if edge.source_id == current_id and edge.target_id not in visited:
                next_id = edge.target_id
                direction = f"--[{edge.relation_type.value}]-->"
            elif edge.target_id == current_id and edge.source_id not in visited:
                next_id = edge.source_id
                direction = f"<--[{edge.relation_type.value}]--"
            
            if next_id:
                visited.add(next_id)
                new_path = f"{path}{direction}" if path else direction
                queue.append((next_id, hops + 1, new_path))
    
    return results


# 示例：查询与"连接池扩容操作"相关的记忆
print("=" * 60)
print("Example: Query Related Memories")
print("=" * 60)

# 找到"连接池扩容操作"的记忆ID
target_memory = None
for m in sample_memories:
    if "扩容" in m.key:
        target_memory = m
        break

if target_memory:
    print(f"\nQuerying memories related to: [{target_memory.key}]")
    related = query_related_memories(memory_graph, target_memory.id, max_hops=2)
    
    print(f"\nFound {len(related)} related memories:")
    for mem_id, hops, path in related:
        mem_key = id_to_key.get(mem_id, mem_id[:8])
        print(f"  [{hops} hop] {mem_key}")
        print(f"           path: {path}")

# %% [markdown]
# ## 总结
# 
# 本notebook实现了记忆组织的四大核心模块：
# 
# 1. **时间结构构建** (TemporalStructureBuilder)
#    - 时间规范化
#    - 会话切分
#    - 阶段抽象
#    - 上下文窗口恢复
# 
# 2. **事件结构构建** (EventStructureBuilder)
#    - 事件要素抽取
#    - 事件关系推断
#    - 事件链构建
# 
# 3. **语义结构与关系建模** (SemanticStructureBuilder)
#    - 语义结构化（topic、entities、slots）
#    - 候选边生成
#    - 关系判别
# 
# 4. **层级化与抽象** (HierarchicalAbstractionBuilder)
#    - 索引树构建
#    - 相似记忆聚合
#    - 模式归纳
#    - 抽象固化
# 
# 通过 `MemoryOrganizer` 可以灵活选择使用哪些模块，最终生成统一的 `MemoryGraph` 结构。

Python: 3.10.18
Platform: macOS-14.6.1-arm64-arm-64bit
Config initialized: MemoryOrganizationConfig(time_threshold_minutes=30, topic_drift_threshold=0.3, context_window_hours=2, event_confidence_threshold=0.5, similarity_threshold=0.3, edge_confidence_threshold=0.5, max_candidates=50, min_cluster_size=2, abstraction_threshold=0.4, verbose=True, use_mock_llm=True)
Generated 8 sample memories:
  [09:00] 网关延迟告警
  [09:05] 初步排查日志
  [09:15] 数据库连接池分析
  [09:25] 连接池扩容操作
  [09:35] 延迟恢复确认
  [10:40] 用户偏好记录
  [12:20] 另一次延迟问题
  [12:40] 内存扩容处理
LLM and Embedding services initialized (mock mode)
Module 1: Temporal Structure Building
  Session break at [09:05]: topic_drift
  Session break at [09:15]: topic_drift
  Session break at [09:25]: topic_drift
  Session break at [09:35]: topic_drift
  Session break at [10:40]: time_gap
  Session break at [12:20]: time_gap
  Session break at [12:40]: topic_drift

Segmented into 8 sessions:
  Session [09:00 - 09:00]: 1 memories, topic: api-gateway
  Session [09:05